# PyTorch ↔ NumPy Fundamentals

Every concept is taught by showing the **NumPy reference** first, then a **PyTorch TODO** to implement,  
followed by an **assertion** that validates your implementation against the reference.

Datasets: `iris` (classification) and `california_housing` (regression) — loaded once, reused everywhere.

In [1]:
# ── Global Imports & Dataset Loading ────────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, fetch_california_housing

# ── Real datasets — used throughout every section ───────────────────────────
iris     = load_iris()            # 150 samples, 4 features, 3 classes
housing  = fetch_california_housing()  # ~20k samples, 8 features

In [ ]:


# Raw NumPy arrays
X_iris   = iris.data.astype(np.float32)        # (150, 4)
y_iris   = iris.target.astype(np.int64)        # (150,)
X_house  = housing.data.astype(np.float32)     # (20640, 8)
y_house  = housing.target.astype(np.float32)   # (20640,)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Iris    X: {X_iris.shape}, y: {y_iris.shape}')
print(f'Housing X: {X_house.shape}, y: {y_house.shape}')

---
## Section 1 — Tensor Creation & Data Loading

`torch.tensor()` copies data into a new tensor; `torch.from_numpy()` **shares memory** (no copy).  
Always specify `dtype` explicitly — implicit promotion can silently lose precision.  
Move tensors to GPU with `.to(device)` — all operations must be on the same device.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_features = np.array(X_iris, dtype=np.float32)
print('dtype:', np_features.dtype, '| shape:', np_features.shape)

np_zeros  = np.zeros((3, 4), dtype=np.float32)
np_ones   = np.ones((3, 4),  dtype=np.float32)
np_eye    = np.eye(4,         dtype=np.float32)
np_arange = np.arange(0, 10, 2, dtype=np.float32)        # [0,2,4,6,8]
np_lin    = np.linspace(0, 1, 5, dtype=np.float32)       # 5 evenly spaced in [0,1]

print('zeros:', np_zeros.shape)
print('arange:', np_arange)
print('linspace:', np_lin)

#### Drill — `torch.tensor vs torch.from_numpy`
Practice the core operation before using it in the problem above.

In [ ]:
x = np.array([1., 2., 3.])
# DRILL: from_numpy shares memory, tensor copies
t_copy = torch.tensor(x)
t_shared = torch.from_numpy(x)
assert t_copy.numel() == 3 and t_shared.numel() == 3

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def create_tensors(X_np: np.ndarray):
    """
    Demonstrate the two ways to get a tensor from a numpy array,
    then recreate the standard factory tensors.

    Returns:
        t_copy      : tensor created with torch.tensor() (data is COPIED)
        t_shared    : tensor created with torch.from_numpy() (memory SHARED)
        zeros, ones, eye, arange, lin : factory tensors matching numpy references
    """
    # Step 1: copy path — creates a new tensor, changes to t_copy won't affect X_np
    t_copy = ...

    # Step 2: shared memory — modifying t_shared WILL modify X_np and vice-versa
    t_shared = ...

    # Step 3: factory functions — match the NumPy reference shapes/dtypes above
    zeros  = ...
    ones   = ...
    eye    = ...
    arange = ...
    lin    = ...

    # Step 4: move features tensor to global device
    t_device = ...

    return t_copy, t_shared, zeros, ones, eye, arange, lin, t_device

(
    t_copy, t_shared, t_zeros, t_ones, t_eye, t_arange, t_lin, t_dev
) = create_tensors(X_iris.copy())

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_copy.dtype  == torch.float32
assert t_copy.shape  == (150, 4)
assert np.allclose(np_zeros,  t_zeros.numpy(),  atol=1e-5)
assert np.allclose(np_ones,   t_ones.numpy(),   atol=1e-5)
assert np.allclose(np_eye,    t_eye.numpy(),    atol=1e-5)
assert np.allclose(np_arange, t_arange.numpy(), atol=1e-5)
assert np.allclose(np_lin,    t_lin.numpy(),    atol=1e-5)
assert str(t_dev.device).startswith(str(device).split(':')[0])
print('Section 1 assertions passed.')

---
## Section 2 — Indexing, Slicing, Masking

Boolean masks let you filter rows without knowing their indices in advance.  
`torch.where(cond, x, y)` is the element-wise conditional equivalent of `np.where`.  
`torch.index_select(dim, index)` gathers specific slices along one dimension.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
# Select all Setosa (class 0) samples
np_setosa        = X_iris[y_iris == 0]                   # (50, 4)

# Select features 0 and 2 only (sepal length + petal length)
np_feat02        = X_iris[:, [0, 2]]                     # (150, 2)

# Boolean mask: housing prices above median
median_price     = np.median(y_house)
np_expensive     = y_house[y_house > median_price]       # ~(10320,)

print('setosa:', np_setosa.shape)
print('feat 0&2:', np_feat02.shape)
print('expensive houses:', np_expensive.shape)

#### Drill — `torch.where`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.tensor([-1, 0, 1])
# DRILL: replace negatives with 0
res = torch.where(x < 0, torch.zeros_like(x), x)
assert res.tolist() == [0, 0, 1]

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def indexing_ops(X: np.ndarray, y: np.ndarray, prices: np.ndarray):
    """
    Reproduce the three NumPy reference operations using PyTorch tensors.

    Args:
        X      : iris features  (150, 4)  float32 numpy
        y      : iris labels    (150,)    int64   numpy
        prices : housing prices (N,)      float32 numpy

    Returns:
        t_setosa   : rows where label == 0,  shape (50, 4)
        t_feat02   : columns 0 and 2,        shape (150, 2)
        t_expensive: prices above median
    """
    tX = torch.tensor(X)       # NumPy equivalent: np.array(X)
    ty = torch.tensor(y)       # NumPy equivalent: np.array(y)
    tp = torch.tensor(prices)  # NumPy equivalent: np.array(prices)

    # Step 1: boolean mask — select Setosa rows
    mask_setosa = ...
    t_setosa    = ...

    # Step 2: index_select — grab columns 0 and 2
    col_idx  = ...
    t_feat02 = ...
                               # NumPy equivalent: X[:, [0, 2]]

    # Step 3: boolean mask on prices
    median   = ...
    t_expensive = ...

    # BONUS Step 4: torch.where — replace prices below median with 0
    # NumPy equivalent: np.where(prices > median, prices, 0)
    t_where = ...

    return t_setosa, t_feat02, t_expensive

t_setosa, t_feat02, t_expensive = indexing_ops(X_iris, y_iris, y_house)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_setosa,  t_setosa.numpy(),    atol=1e-5), 'setosa mismatch'
assert np.allclose(np_feat02,  t_feat02.numpy(),    atol=1e-5), 'feat02 mismatch'
assert np.allclose(np_expensive, t_expensive.numpy(), atol=1e-5), 'expensive mismatch'
assert t_setosa.shape   == (50, 4)
assert t_feat02.shape   == (150, 2)
print('Section 2 assertions passed.')

---
## Section 3 — Reshaping & Dimension Operations

`.view()` requires **contiguous** memory; `.reshape()` will make a copy if needed — prefer `.reshape()` unless you want the guarantee.  
`.unsqueeze(dim)` adds a size-1 dimension; `.squeeze(dim)` removes it.  
`torch.cat` joins along an existing axis; `torch.stack` creates a **new** axis.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_X = X_iris.copy()                             # (150, 4)

np_expanded = np.expand_dims(np_X, axis=2)       # (150, 4, 1)
np_squeezed = np.squeeze(np_expanded, axis=2)    # (150, 4)
np_T        = np_X.T                             # (4, 150)  — transpose

# Stack two copies side by side along axis 0 → (300, 4)
np_stacked  = np.concatenate([np_X, np_X], axis=0)

# Stack as a new dimension → (2, 150, 4)
np_newdim   = np.stack([np_X, np_X], axis=0)

print('expanded:', np_expanded.shape)
print('squeezed:', np_squeezed.shape)
print('transposed:', np_T.shape)
print('cat along 0:', np_stacked.shape)
print('stack new dim:', np_newdim.shape)

#### Drill — `torch.stack vs torch.cat`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.ones(2, 2)
# DRILL: stack creates new dim, cat joins along existing
stacked = torch.stack([x, x], dim=0)
cat = torch.cat([x, x], dim=0)
assert stacked.shape == (2, 2, 2) and cat.shape == (4, 2)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def reshape_ops(X_np: np.ndarray):
    """
    Reproduce all five NumPy reference reshaping ops.

    Returns:
        t_expanded  : (150, 4, 1)
        t_squeezed  : (150, 4)
        t_T         : (4, 150)
        t_cat       : (300, 4)  — torch.cat along dim 0
        t_stack     : (2, 150, 4) — torch.stack adding new dim
    """
    tX = torch.tensor(X_np)   # (150, 4)

    # Step 1: add size-1 dim at position 2
    # NumPy equivalent: np.expand_dims(X, axis=2)
    t_expanded = ...

    # Step 2: remove that size-1 dim
    # NumPy equivalent: np.squeeze(X, axis=2)
    t_squeezed = ...

    # Step 3a: .view() — ONLY works if tensor is contiguous
    # Demonstrate: transpose makes it non-contiguous
    t_noncontig = tX.permute(1, 0)   # (4, 150) — non-contiguous after permute
    # t_noncontig.view(4, 150)  ← would raise RuntimeError
    # Fix: call .contiguous() first
    t_contig    = ...
    t_view_ok   = ...

    # Step 3b: .permute() vs .transpose()
    # NumPy equivalent: X.T
    t_T         = ...

    # Step 4: torch.cat — joins along existing dimension
    # NumPy equivalent: np.concatenate([X, X], axis=0)
    t_cat       = ...

    # Step 5: torch.stack — creates a NEW dimension
    # NumPy equivalent: np.stack([X, X], axis=0)
    t_stack     = ...

    return t_expanded, t_squeezed, t_T, t_cat, t_stack

t_expanded, t_squeezed, t_T, t_cat, t_stack = reshape_ops(X_iris)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_expanded.shape == (150, 4, 1)
assert t_squeezed.shape == (150, 4)
assert np.allclose(np_T,       t_T.numpy(),      atol=1e-5)
assert np.allclose(np_stacked, t_cat.numpy(),    atol=1e-5)
assert np.allclose(np_newdim,  t_stack.numpy(),  atol=1e-5)
print('Section 3 assertions passed.')

---
## Section 4 — Math & Broadcasting

Broadcasting aligns shapes from the **right**: dimensions of size 1 are stretched to match.  
`torch.matmul` handles 1-D through N-D; `torch.mm` is 2-D only; `torch.bmm` is batched 3-D.  
Normalisation is the most common broadcasting pattern you will write in every model.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_X  = X_iris.copy()                                    # (150, 4)

# Feature normalisation with broadcasting
np_mean  = np_X.mean(axis=0)                             # (4,)
np_std   = np_X.std(axis=0)                              # (4,)
np_norm  = (np_X - np_mean) / (np_std + 1e-8)           # (150, 4)

# Dot product: (150,4) @ (4,1) → (150,1) — project onto a weight vector
np_W     = np.ones((4, 1), dtype=np.float32)
np_proj  = np_X @ np_W                                   # (150, 1)

print('normalised mean (should ≈ 0):', np_norm.mean(axis=0).round(4))
print('projection shape:', np_proj.shape)

#### Drill — `Broadcasting (N,1) vs (1,M)`
Practice the core operation before using it in the problem above.

In [ ]:
A = torch.tensor([[1], [2], [3]])
B = torch.tensor([[10, 20]])
# DRILL: outer sum
res = A + B
assert res.shape == (3, 2) and res[0].tolist() == [11, 21]

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def math_ops(X_np: np.ndarray):
    """
    Reproduce normalisation and matrix projection using PyTorch.

    Returns:
        t_norm  : (150, 4)  — normalised features, should match np_norm
        t_proj  : (150, 1)  — projection onto all-ones weight vector
        t_clamp : (150, 4)  — normalised features clamped to [-2, 2]
    """
    tX = torch.tensor(X_np)   # (150, 4)

    # Step 1: compute per-feature mean and std along dim 0
    # NumPy equivalent: X.mean(axis=0), X.std(axis=0)
    t_mean = ...
    t_std  = ...
                               # NOTE: torch default is unbiased=True (Bessel), numpy default is biased
                               #       use unbiased=False to match numpy

    # Step 2: normalise — broadcasting automatically aligns (150,4) - (4,)
    t_norm = ...

    # Step 3: matrix multiplication variants
    tW = torch.ones(4, 1)     # weight vector
    # torch.mm  — 2D only; NumPy equivalent: np.matmul or @
    t_proj_mm     = ...
    t_proj_matmul = ...
    t_batch_mm = ...
                               # → (150, 1, 1)

    # Step 4: element-wise ops
    # NumPy equivalent: np.clip(x, -2, 2)
    t_clamp = ...

    # torch.abs, torch.exp, torch.log (applied to small tensor for clarity)
    sample = t_norm[0]         # shape (4,)
    _ = torch.abs(sample)      # NumPy equivalent: np.abs
    _ = torch.exp(sample)      # NumPy equivalent: np.exp
    _ = torch.log(torch.abs(sample) + 1e-8)  # NumPy equivalent: np.log

    return t_norm, t_proj_mm, t_clamp

t_norm, t_proj, t_clamp = math_ops(X_iris)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_norm, t_norm.numpy(), atol=1e-4), 'normalisation mismatch'
assert np.allclose(np_proj, t_proj.numpy(), atol=1e-4), 'projection mismatch'
assert t_clamp.numpy().min() >= -2 - 1e-5
assert t_clamp.numpy().max() <=  2 + 1e-5
print('Section 4 assertions passed.')

---
## Section 5 — Reductions & Statistics

Pass `dim=` to reduce along a specific axis; omit it to reduce the whole tensor.  
`torch.topk(k)` returns a named tuple `(values, indices)` — useful for top-k accuracy.  
`torch.cumsum` produces running totals and is handy for masking variable-length sequences.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_X = X_iris.copy()          # (150, 4)

np_feat_mean = np_X.mean(axis=0)      # (4,) — per feature
np_feat_std  = np_X.std(axis=0)       # (4,)
np_feat_min  = np_X.min(axis=0)       # (4,)
np_feat_max  = np_X.max(axis=0)       # (4,)

np_argmax    = np.argmax(np_X, axis=1)          # (150,) — which feature is largest per sample
np_pct_90    = np.percentile(y_house, 90)        # scalar — 90th percentile of housing prices

print('per-feature mean:', np_feat_mean)
print('90th pct price:', np_pct_90)

#### Drill — `torch.topk`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.tensor([10., 30., 20., 50., 40.])
# DRILL: top 2 values
vals, idx = torch.topk(x, k=2)
assert vals.tolist() == [50., 40.] and idx.tolist() == [3, 4]

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def reduction_ops(X_np: np.ndarray, prices_np: np.ndarray):
    """
    Implement all reduction operations using PyTorch.

    Returns:
        t_mean, t_std, t_min, t_max : per-feature stats, each (4,)
        t_argmax   : (150,) — argmax per sample
        topk_vals  : top-3 feature values per sample (150, 3)
        topk_idx   : corresponding indices           (150, 3)
        t_unique   : unique class counts
        t_cumsum   : cumulative sum along feature 0
    """
    tX = torch.tensor(X_np)            # (150, 4)
    tp = torch.tensor(prices_np)       # (N,)

    # Step 1: per-feature statistics — reduce along dim 0 (across samples)
    # NumPy equivalent: X.mean(axis=0)
    t_mean = ...
    t_std  = ...
    t_min  = ...
    t_max  = ...

    # Step 2: argmax — which feature is largest for each sample?
    # NumPy equivalent: np.argmax(X, axis=1)
    t_argmax = ...

    # Step 3: top-k — top 3 feature values (and their indices) per sample
    # NumPy equivalent: np.argsort(X, axis=1)[:, -3:]  (but topk is faster)
    topk_result = ...
    topk_vals   = ...
    topk_idx    = ...

    # Step 4: unique values in price tensor (torch.unique)
    # NumPy equivalent: np.unique(prices, return_counts=True)
    t_unique = ...

    # Step 5: cumulative sum along feature 0 for all 150 samples
    # NumPy equivalent: np.cumsum(X[:, 0])
    t_cumsum = ...

    return t_mean, t_std, t_min, t_max, t_argmax, topk_vals, topk_idx, t_unique, t_cumsum

(
    t_mean, t_std, t_min, t_max,
    t_argmax, topk_vals, topk_idx,
    t_unique, t_cumsum
) = reduction_ops(X_iris, y_house)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_feat_mean, t_mean.numpy(),   atol=1e-4)
assert np.allclose(np_feat_std,  t_std.numpy(),    atol=1e-4)
assert np.allclose(np_feat_min,  t_min.numpy(),    atol=1e-4)
assert np.allclose(np_feat_max,  t_max.numpy(),    atol=1e-4)
assert np.array_equal(np_argmax, t_argmax.numpy())
assert topk_vals.shape  == (150, 3)
assert topk_idx.shape   == (150, 3)
assert t_cumsum.shape   == (150,)
print('Section 5 assertions passed.')

---
## Section 6 — Linear Algebra

SVD factorises **X = U Σ Vᵀ**. The top-k singular vectors form the best rank-k approximation.  
PCA = centre data → SVD → project onto top eigenvectors (the rows of Vᵀ).  
`torch.linalg` mirrors `numpy.linalg` almost 1-to-1 since PyTorch 1.9.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_X     = X_iris.copy().astype(np.float64)   # use float64 for linalg stability

# Covariance matrix
np_Xc    = np_X - np_X.mean(axis=0)           # centred (150, 4)
np_cov   = (np_Xc.T @ np_Xc) / (len(np_Xc) - 1)  # (4, 4)

# SVD of centred data
np_U, np_S, np_Vt = np.linalg.svd(np_Xc, full_matrices=False)
# np_U: (150, 4), np_S: (4,), np_Vt: (4, 4)

# PCA: project onto top 2 components
np_pca2  = np_Xc @ np_Vt.T[:, :2]            # (150, 2)

# Norm and inverse of cov matrix
np_norm_cov = np.linalg.norm(np_cov, ord='fro')   # Frobenius norm
np_inv_cov  = np.linalg.inv(np_cov)               # (4, 4)

print('covariance shape:', np_cov.shape)
print('SVD: U', np_U.shape, 'S', np_S.shape, 'Vt', np_Vt.shape)
print('PCA projection:', np_pca2.shape)
print('Frobenius norm:', np_norm_cov)

#### Drill — `torch.linalg.svd`
Practice the core operation before using it in the problem above.

In [ ]:
A = torch.randn(4, 3)
# DRILL: economy SVD
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
assert U.shape == (4, 3) and S.shape == (3,) and Vh.shape == (3, 3)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def linalg_ops(X_np: np.ndarray):
    """
    Reproduce the NumPy linear algebra reference using torch.linalg.

    Returns:
        t_cov    : (4, 4)   covariance matrix
        t_S      : (4,)     singular values
        t_pca2   : (150, 2) PCA projection onto top 2 components
        t_norm   : scalar   Frobenius norm of covariance
        t_inv    : (4, 4)   inverse of covariance
    """
    tX = torch.tensor(X_np, dtype=torch.float64)   # use float64 for stability

    # Step 1: centre the data
    # NumPy equivalent: X - X.mean(axis=0)
    t_Xc = ...

    # Step 2: covariance matrix
    t_cov = ...

    # Step 3: SVD — torch.linalg.svd returns (U, S, Vh) where Vh = Vᵀ
    # NumPy equivalent: np.linalg.svd(X, full_matrices=False)
    t_U, t_S, t_Vh = ...

    # Step 4: PCA projection onto top 2 components
    # Vt rows = principal directions; project: Xc @ Vt.T[:, :2]
    # NumPy equivalent: Xc @ Vt.T[:, :2]
    t_pca2 = ...

    # Step 5: Frobenius norm of covariance
    # NumPy equivalent: np.linalg.norm(cov, ord='fro')
    t_norm = ...

    # Step 6: matrix inverse
    # NumPy equivalent: np.linalg.inv(cov)
    t_inv  = ...

    # BONUS: eigenvalues of symmetric covariance matrix
    # NumPy equivalent: np.linalg.eig(cov)
    t_eigvals, t_eigvecs = ...

    return t_cov, t_S, t_pca2, t_norm, t_inv

t_cov, t_S, t_pca2, t_fnorm, t_inv = linalg_ops(X_iris.astype(np.float64))

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_cov,      t_cov.numpy(),   atol=1e-5), 'cov mismatch'
assert np.allclose(np_S,        t_S.numpy(),     atol=1e-5), 'singular values mismatch'
# PCA signs can flip — compare absolute values
assert np.allclose(np.abs(np_pca2), np.abs(t_pca2.numpy()), atol=1e-4), 'PCA mismatch'
assert np.allclose(np_norm_cov, float(t_fnorm),  atol=1e-4), 'norm mismatch'
assert np.allclose(np_inv_cov,  t_inv.numpy(),   atol=1e-4), 'inv mismatch'

# Quick visualisation of PCA
plt.figure(figsize=(6,4))
pca_np = t_pca2.numpy()
for cls, name in enumerate(iris.target_names):
    mask = y_iris == cls
    plt.scatter(pca_np[mask,0], pca_np[mask,1], label=name, alpha=0.7)
plt.title('PCA of Iris (top 2 components via SVD)')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.tight_layout(); plt.show()
print('Section 6 assertions passed.')

---
## Section 7 — Autograd: The Core of PyTorch ⚡

PyTorch builds a **dynamic computational graph** as you execute operations on tensors with `requires_grad=True`.  
Calling `.backward()` walks this graph in reverse (chain rule) and accumulates gradients into `.grad`.  
**Always call `optimizer.zero_grad()` before each backward** — gradients accumulate by default (they don't reset).

### Math: MSE Gradient by Hand

Given $\hat{y} = X W$, the MSE loss is:
$$L = \frac{1}{N} \sum_i (\hat{y}_i - y_i)^2$$

Gradient w.r.t. $W$:
$$\frac{\partial L}{\partial W} = \frac{2}{N} X^T (\hat{y} - y)$$

In [ ]:
# ── NumPy Reference: Manual Gradient ─────────────────────────────────────────
# Use iris features (150,4) to predict a scalar target y
# For demonstration we use y_iris cast to float as the target
np_X_ag = X_iris.copy().astype(np.float64)     # (150, 4)
np_y_ag = y_iris.astype(np.float64).reshape(-1, 1)  # (150, 1)

# Fixed random weights (same seed for reproducibility)
rng = np.random.default_rng(42)
np_W = rng.standard_normal((4, 1)).astype(np.float64) * 0.01   # (4, 1)

# Forward pass
np_y_pred = np_X_ag @ np_W                     # (150, 1)
np_diff   = np_y_pred - np_y_ag                # (150, 1)
np_loss   = (np_diff ** 2).mean()              # scalar

# Analytical gradient: dL/dW = (2/N) * X.T @ (y_pred - y)
N = len(np_X_ag)
np_dW = (2 / N) * (np_X_ag.T @ np_diff)       # (4, 1)

print('NumPy loss:', round(float(np_loss), 6))
print('NumPy dW (first 4 values):', np_dW.flatten())

#### Drill — `loss.backward()`
Practice the core operation before using it in the problem above.

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])
# DRILL: gradient of x*w^2 wrt w
loss = x * w**2
loss.backward()
assert w.grad.item() == 12.0

In [ ]:
# ── PyTorch TODO — Autograd ───────────────────────────────────────────────────
def autograd_demo(X_np, y_np, W_np):
    """
    Replicate the manual gradient using PyTorch autograd.
    The autograd result MUST match the analytical NumPy gradient.

    Args:
        X_np : (150, 4) float64 numpy features
        y_np : (150, 1) float64 numpy targets
        W_np : (4,  1)  float64 numpy weights (same as used in NumPy reference)

    Returns:
        loss  : scalar tensor (detached float)
        W_grad: (4, 1) tensor — the autograd-computed gradient
    """
    # Step 1: convert to tensors; W MUST have requires_grad=True
    tX = torch.tensor(X_np)
    ty = torch.tensor(y_np)
    # NumPy equivalent: no direct equivalent — this marks W for gradient tracking
    tW = ...

    # Step 2: forward pass — same math as numpy
    # NumPy equivalent: y_pred = X @ W
    t_y_pred = ...

    # Step 3: MSE loss — (y_pred - y)^2 mean
    # NumPy equivalent: ((y_pred - y)**2).mean()
    t_loss = ...

    # Step 4: backward pass — compute all gradients
    # This walks the computational graph in reverse (chain rule)
    ...

    # Step 5: inspect the gradient
    # tW.grad now holds dL/dW — populated by backward()
    W_grad = ...     # tW.grad   ← shape (4, 1)

    # ── When to use torch.no_grad() ──────────────────────────────────────────
    # During inference or validation you don't need gradients — skip graph building
    # NumPy equivalent: no graph is ever built in NumPy
    with torch.no_grad():
        t_inference = tX @ tW   # no graph tracked — faster, less memory

    # ── .detach() use case ───────────────────────────────────────────────────
    # Detaches tensor from graph so it won't be part of future backward passes
    t_detached = t_loss.detach()   # now a plain tensor, not part of graph

    return float(t_loss.detach()), W_grad

pt_loss, pt_dW = autograd_demo(np_X_ag, np_y_ag, np_W)

In [ ]:
# ── Assertions — most critical check in the notebook ─────────────────────────
print('NumPy   loss:', round(float(np_loss), 8))
print('PyTorch loss:', round(float(pt_loss), 8))
print('NumPy   dW:', np_dW.flatten())
print('PyTorch dW:', pt_dW.detach().numpy().flatten())

assert np.allclose(float(np_loss), float(pt_loss), atol=1e-6), 'loss mismatch'
# THE key assertion: autograd must match hand-derived gradient
assert np.allclose(np_dW, pt_dW.detach().numpy(), atol=1e-5), \
    f'GRADIENT MISMATCH\nNumPy: {np_dW.flatten()}\nPyTorch: {pt_dW.flatten()}'

print('\nSection 7 assertions passed — autograd matches analytical gradient.')

---
## Section 8 — Linear Model from Scratch (No `nn.Linear`)

This is the raw training loop: **forward → loss → backward → manual weight update**.  
You must zero the gradient **yourself** with `W.grad.zero_()` — there is no optimizer to do it.  
Update weights via `.data -=` (not `-=`) to avoid creating a new graph node inside `.grad`.

In [ ]:
# ── PyTorch TODO — Manual Training Loop ──────────────────────────────────────
def train_manual(X_np, y_np, lr=0.001, epochs=100, print_every=10):
    """
    Train a linear regression model (y = XW + b) on the housing dataset
    without using any nn.Module or optimizer objects.

    Args:
        X_np       : (N, 8) float32 numpy — housing features
        y_np       : (N,)   float32 numpy — housing targets
        lr         : learning rate
        epochs     : number of gradient descent steps
        print_every: print loss interval

    Returns:
        losses : list of floats — loss at each epoch
        W, b   : final trained weight tensors
    """
    tX = torch.tensor(X_np)            # (N, 8)
    ty = torch.tensor(y_np).unsqueeze(1)  # (N, 1)

    # Step 1: initialise weights with small random values
    # MUST have requires_grad=True so autograd tracks operations on them
    # NumPy equivalent: W = np.random.randn(8, 1) * 0.01
    torch.manual_seed(42)
    W = ...
    b = ...

    losses = []
    for epoch in range(epochs):
        # Step 2: forward pass
        # NumPy equivalent: y_pred = X @ W + b
        y_pred = ...

        # Step 3: MSE loss
        # NumPy equivalent: ((y_pred - y)**2).mean()
        loss = ...

        # Step 4: backward — compute dL/dW and dL/db
        ...

        # Step 5: manual gradient descent — update weights in-place
        # Use .data to avoid creating a new graph node
        # NumPy equivalent: W -= lr * dW
        with torch.no_grad():
            ...
            ...

        # Step 6: MUST zero gradients before next backward!
        # (PyTorch accumulates gradients — doesn't reset them automatically)
        ...
        ...

        losses.append(float(loss.detach()))
        if (epoch + 1) % print_every == 0:
            print(f'Epoch {epoch+1:3d} | loss: {losses[-1]:.4f}')

    return losses, W, b

# Use only first 2000 samples for speed
losses_manual, W_final, b_final = train_manual(
    X_house[:2000], y_house[:2000], lr=0.001, epochs=100
)

In [ ]:
# ── Loss Curve ────────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(losses_manual, label='Manual SGD')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Section 8 — Manual Training Loop (Housing Dataset)')
plt.legend(); plt.tight_layout(); plt.show()

assert losses_manual[-1] < losses_manual[0], 'Loss should decrease over training'
print('Section 8 assertions passed.')

---
## Section 9 — Building with `nn.Module`

`nn.Module` abstracts weight management, device placement, and parameter enumeration.  
The training loop structure is: `zero_grad → forward → loss → backward → step`.  
SGD updates via $W \leftarrow W - \eta \nabla L$; Adam adds adaptive per-parameter learning rates (usually converges faster).

In [ ]:
import torch.nn as nn
import torch.optim as optim

# ── PyTorch TODO — nn.Module ──────────────────────────────────────────────────
class LinearRegressor(nn.Module):
    """
    Simple linear regressor: y = ReLU(hidden) → linear → scalar output.
    Architecture: Linear(8→16) → ReLU → Linear(16→1)
    """
    def __init__(self, in_features: int = 8, hidden: int = 16):
        super().__init__()
        # Step 1: define layers as attributes
        # NumPy equivalent: W1 = randn(8,16), b1 = zeros(16), etc.
        self.fc1  = ...
        self.relu = ...
        self.fc2  = ...

        # Alternative: nn.Sequential packs layers into one callable
        # self.net = nn.Sequential(
        #     nn.Linear(in_features, hidden),
        #     nn.ReLU(),
        #     nn.Linear(hidden, 1)
        # )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:    x : (B, 8)
        Returns: y : (B, 1)
        """
        # Step 2: chain the layers
        # NumPy equivalent: x = relu(x @ W1 + b1); return x @ W2 + b2
        x = ...
        x = ...
        x = ...
        return x


def train_nn(X_np, y_np, optimizer_type='adam', lr=1e-3, epochs=100, print_every=10):
    """
    Full training loop using nn.Module, a loss function, and an optimizer.

    Returns:
        model  : trained LinearRegressor
        losses : list[float]
    """
    tX = torch.tensor(X_np)
    ty = torch.tensor(y_np).unsqueeze(1)

    model = LinearRegressor(in_features=X_np.shape[1])

    # Step 3: pick loss function
    # For regression:       nn.MSELoss()
    # For classification:   nn.CrossEntropyLoss()  (logits in, class indices target)
    criterion = ...

    # Step 4: pick optimizer — model.parameters() returns all nn.Module weights
    if optimizer_type == 'sgd':
        optimizer = ...
    else:
        optimizer = ...

    losses = []
    for epoch in range(epochs):
        # The canonical 4-step training loop:
        ...              # 1. optimizer.zero_grad()   — clear old gradients
        y_pred = ...     # 2. model(tX)               — forward pass
        loss   = ...     # 3. criterion(y_pred, ty)   — compute loss
        ...              # 4. loss.backward()         — backward pass
        ...

        losses.append(float(loss.detach()))
        if (epoch + 1) % print_every == 0:
            print(f'[{optimizer_type.upper()}] Epoch {epoch+1:3d} | loss: {losses[-1]:.4f}')

    return model, losses


model_sgd,  losses_sgd  = train_nn(X_house[:2000], y_house[:2000], optimizer_type='sgd',  epochs=100)
model_adam, losses_adam = train_nn(X_house[:2000], y_house[:2000], optimizer_type='adam', epochs=100)

In [ ]:
# ── SGD vs Adam Comparison Plot ───────────────────────────────────────────────
plt.figure(figsize=(8, 3))
plt.plot(losses_sgd,  label='SGD')
plt.plot(losses_adam, label='Adam')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Section 9 — SGD vs Adam (Housing Dataset)')
plt.legend(); plt.tight_layout(); plt.show()

assert losses_adam[-1] < losses_adam[0], 'Adam loss should decrease'
assert losses_sgd[-1]  < losses_sgd[0],  'SGD loss should decrease'
print('Section 9 assertions passed.')

---
## Section 10 — Dataset and DataLoader

A `Dataset` defines **what** data you have (`__len__`, `__getitem__`).  
A `DataLoader` defines **how** you iterate — batching, shuffling, parallel workers.  
Mini-batch training is almost always faster than full-batch because gradient updates happen more frequently.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# ── PyTorch TODO — Custom Dataset ────────────────────────────────────────────
class IrisDataset(Dataset):
    """
    Wraps the Iris numpy arrays in a PyTorch Dataset.

    __getitem__ must return a single (features, label) pair.
    DataLoader will batch these pairs together automatically.
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Step 1: store as tensors
        self.X = ...
        self.y = ...

    def __len__(self) -> int:
        # Step 2: return number of samples
        return ...   # len(self.X)

    def __getitem__(self, idx: int):
        # Step 3: return a single (features, label) pair
        return ...   # self.X[idx], self.y[idx]


# ── DataLoader ────────────────────────────────────────────────────────────────
iris_dataset = IrisDataset(X_iris, y_iris)

# Step 4: wrap in DataLoader
# batch_size=16: each iteration yields 16 samples
# shuffle=True:  randomise order each epoch (prevents memorising order)
loader = ...

# Step 5: inspect one batch
for batch_X, batch_y in loader:
    print('batch X shape:', batch_X.shape)   # expect (16, 4) or smaller for last batch
    print('batch y shape:', batch_y.shape)   # expect (16,)
    break

#### Drill — `Training Step`
Practice the core operation before using it in the problem above.

In [ ]:
w = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)
# DRILL: standard training step
optimizer.zero_grad()
loss = (w - 3.0)**2
loss.backward()
optimizer.step()
assert w.item() == 1.4

In [ ]:
# ── PyTorch TODO — Classifier for Iris ───────────────────────────────────────
class IrisClassifier(nn.Module):
    """
    3-class classifier for Iris.
    Architecture: Linear(4→16) → ReLU → Linear(16→3)
    Output: raw logits (CrossEntropyLoss handles softmax internally)
    """
    def __init__(self):
        super().__init__()
        self.net = ...

    def forward(self, x):
        return ...  # self.net(x)


def train_with_loader(loader, epochs=30):
    """Train IrisClassifier using the DataLoader."""
    model     = IrisClassifier()
    criterion = ...
    optimizer = ...
    losses    = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_X, batch_y in loader:          # iterate over all mini-batches
            optimizer.zero_grad()
            logits = ...
            loss   = ...
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.detach())
        losses.append(epoch_loss / len(loader))

    return model, losses

clf_model, clf_losses = train_with_loader(loader, epochs=30)
print(f'Final classification loss: {clf_losses[-1]:.4f}')

assert clf_losses[-1] < clf_losses[0], 'Classifier loss should decrease'
print('Section 10 assertions passed.')

---
## Section 11 — Train / Val Split and Evaluation

Never use test data during training or hyperparameter selection — it gives an optimistic bias.  
Wrap validation code in `torch.no_grad()` — it disables graph building, reducing memory by ~30%.  
Accuracy = fraction of predictions that match the ground truth label.

#### Drill — `Validation Step (no_grad)`
Practice the core operation before using it in the problem above.

In [ ]:
w = torch.tensor([1.0], requires_grad=True)
# DRILL: operations inside no_grad do not track history
with torch.no_grad():
    res = w * 2
assert res.requires_grad is False

In [ ]:
from sklearn.model_selection import train_test_split

# ── PyTorch TODO — Train/Val Split + Evaluation Loop ─────────────────────────
def train_val_loop(X_np, y_np, val_size=0.2, epochs=50, lr=1e-3):
    """
    Full pipeline: split data → train → evaluate each epoch → plot curves.

    Returns:
        model          : trained IrisClassifier
        train_losses   : list[float]
        val_losses     : list[float]
        val_accuracy   : float — accuracy on validation set after training
    """
    # Step 1: split into train / val
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_np, y_np, test_size=val_size, random_state=42, stratify=y_np
    )

    # Step 2: create DataLoaders
    train_loader = DataLoader(IrisDataset(X_tr,  y_tr),  batch_size=16, shuffle=True)
    val_loader   = DataLoader(IrisDataset(X_val, y_val), batch_size=16, shuffle=False)

    model     = IrisClassifier()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # ── Training phase ────────────────────────────────────────────────
        model.train()                  # enable dropout/batchnorm training behaviour
        t_loss = 0.0
        for bX, by in train_loader:
            ...
            optimizer.zero_grad()
            logits = ...
            loss   = ...
            loss.backward()
            optimizer.step()
            t_loss += float(loss.detach())
        train_losses.append(t_loss / len(train_loader))

        # ── Validation phase ──────────────────────────────────────────────
        model.eval()                   # disable dropout/batchnorm training behaviour
        v_loss = 0.0
        with torch.no_grad():          # no gradient tracking during eval
            for bX, by in val_loader:
                logits = ...
                loss   = ...
                v_loss += float(loss)
        val_losses.append(v_loss / len(val_loader))

    # Step 3: compute final accuracy on validation set
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for bX, by in val_loader:
            logits = ...
            preds  = ...
            correct += (preds == by).sum().item()
            total   += len(by)
    val_accuracy = correct / total

    return model, train_losses, val_losses, val_accuracy


model_cv, tr_losses, val_losses, val_acc = train_val_loop(X_iris, y_iris, epochs=60)

In [ ]:
# ── Train vs Val Loss Plot + Assertions ───────────────────────────────────────
plt.figure(figsize=(8, 3))
plt.plot(tr_losses,  label='Train loss')
plt.plot(val_losses, label='Val loss',  linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.title(f'Section 11 — Train/Val Curves  (val accuracy={val_acc:.2%})')
plt.legend(); plt.tight_layout(); plt.show()

assert val_acc > 0.6, f'Expected val accuracy > 60%, got {val_acc:.2%}'
assert tr_losses[-1] < tr_losses[0], 'Train loss should decrease'
print(f'Validation accuracy: {val_acc:.2%}')
print('Section 11 assertions passed.')

---
## Section 12 — Saving and Loading Models

`state_dict()` is an `OrderedDict` of parameter tensors — the canonical checkpoint format.  
Always call `model.load_state_dict()` after creating a fresh model instance (not just `torch.load`).  
Verify that predictions before and after serialisation are bit-identical.

#### Drill — `Model State Dict`
Practice the core operation before using it in the problem above.

In [ ]:
layer = torch.nn.Linear(2, 1)
# DRILL: extract state dict
sd = layer.state_dict()
assert 'weight' in sd and 'bias' in sd

In [ ]:
import os, tempfile

# ── PyTorch TODO — Save & Load ────────────────────────────────────────────────
def save_and_reload(model: nn.Module, X_np: np.ndarray):
    """
    Save model state_dict, reload into a fresh instance, verify predictions match.

    Args:
        model  : any trained nn.Module
        X_np   : input features for verification (numpy)

    Returns:
        preds_before : (N, C) predictions before save
        preds_after  : (N, C) predictions after reload — must match
    """
    tX = torch.tensor(X_np, dtype=torch.float32)

    # Step 1: get predictions before saving
    model.eval()
    with torch.no_grad():
        preds_before = ...

    # Step 2: save state_dict to a temp file
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        ckpt_path = f.name
    ...

    # Step 3: create a FRESH model and load weights
    new_model = IrisClassifier()
    ...
    new_model.eval()

    # Step 4: predictions after reload
    with torch.no_grad():
        preds_after = ...

    os.unlink(ckpt_path)     # clean up temp file
    return preds_before, preds_after


pb, pa = save_and_reload(model_cv, X_iris)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(pb.numpy(), pa.numpy(), atol=1e-6), \
    'Predictions changed after save/load!'
print('Predictions identical before and after checkpoint.')
print('Section 12 assertions passed.')

---
## Section 13 — Tensor Memory & Performance

`view()` requires a **contiguous** memory layout; `permute`/`transpose` break contiguity.  
In-place ops (e.g., `add_()`) are memory-efficient but **break autograd** if the operand is part of a graph.  
`pin_memory()` allocates page-locked (pinned) RAM, enabling asynchronous CPU→GPU transfer via `.to(device, non_blocking=True)`.

#### Drill — `Contiguous memory`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.zeros(2, 3)
# DRILL: transpose loses contiguity
x_t = x.transpose(0, 1)
assert not x_t.is_contiguous()
x_c = x_t.contiguous()
assert x_c.is_contiguous()

In [ ]:
# ── PyTorch TODO — Memory & Performance ──────────────────────────────────────
def memory_demo():
    """
    Demonstrate contiguity issues, in-place ops, and pinned memory.
    """
    tX = torch.tensor(X_iris)           # (150, 4)  contiguous

    # ── 1. Contiguity ────────────────────────────────────────────────────────
    t_T = tX.transpose(0, 1)            # (4, 150) — NON-contiguous after transpose
    print('is_contiguous after transpose:', t_T.is_contiguous())  # False

    try:
        # Step 1a: this WILL raise RuntimeError — view requires contiguous storage
        _ = t_T.view(600)               # 4*150=600 elements
    except RuntimeError as e:
        print('Expected error:', str(e)[:80])

    # Step 1b: fix with .contiguous() then .view()
    t_T_contig = ...
    t_flat      = ...
    print('After .contiguous().view():', t_flat.shape)  # expect (600,)

    # ── 2. In-place ops ──────────────────────────────────────────────────────
    t_safe = torch.tensor(X_iris)     # no requires_grad — in-place is safe
    # NumPy equivalent: X += 1.0
    ...
    ...
    print('In-place ops applied (no autograd graph here — safe).')

    # In-place is DANGEROUS when tensor has requires_grad
    t_grad = torch.tensor(X_iris, requires_grad=True)
    try:
        # Step 2b: autograd records the state BEFORE the in-place op
        # Mutating it corrupts the backward pass
        t_grad.add_(1.0)               # ← will raise RuntimeError in most cases
    except RuntimeError as e:
        print('In-place on leaf with grad expected error:', str(e)[:80])

    # ── 3. GPU memory reporting ───────────────────────────────────────────────
    if torch.cuda.is_available():
        t_gpu = torch.tensor(X_iris).to('cuda')
        print('CUDA memory allocated (bytes):', torch.cuda.memory_allocated())
    else:
        print('CUDA not available — skipping GPU memory demo.')

    # ── 4. Pinned memory ─────────────────────────────────────────────────────
    # pin_memory() allocates page-locked memory → faster async CPU→GPU transfers
    # Typically used inside a DataLoader: DataLoader(..., pin_memory=True)
    if torch.cuda.is_available():
        t_pinned = torch.tensor(X_iris).pin_memory()
        print('is_pinned:', t_pinned.is_pinned())  # True
        # Transfer to GPU with non-blocking async copy
        t_pinned_gpu = t_pinned.to('cuda', non_blocking=True)


memory_demo()
print('Section 13 complete.')

---
## Quick Reference Card

| NumPy | PyTorch | Notes |
|---|---|---|
| `np.array(x)` | `torch.tensor(x)` | copies data |
| — | `torch.from_numpy(x)` | **shares** memory |
| `x.reshape(...)` | `x.reshape(...)` or `x.view(...)` | view needs contiguous |
| `np.expand_dims(x,ax)` | `x.unsqueeze(dim)` | add size-1 dim |
| `np.squeeze(x,ax)` | `x.squeeze(dim)` | remove size-1 dim |
| `np.concatenate` | `torch.cat` | existing dim |
| `np.stack` | `torch.stack` | new dim |
| `x.T` / `np.transpose` | `x.T` / `x.permute(...)` | |
| `np.matmul` / `@` | `torch.matmul` / `@` | N-D |
| `np.dot` | `torch.mm` | 2-D only |
| — | `torch.bmm` | batched 3-D |
| `x.mean(axis=)` | `x.mean(dim=)` | |
| `np.argmax(x,ax)` | `torch.argmax(x,dim=)` | |
| `np.clip` | `torch.clamp` | |
| `np.linalg.svd` | `torch.linalg.svd` | |
| — | `loss.backward()` | builds grad |
| — | `optimizer.zero_grad()` | always before backward |
| — | `torch.no_grad()` | inference/eval |
| — | `.detach()` | stop gradient flow |